# Exercise 3: Case and word order

*The Computational Phenotype · version 0.1.0*

> Translate *Kitabı Ayşe okudu* and *Ayşe kitabı okudu* into English. Give a question that each order would answer naturally. Then compare *Ayşe kitap okudu*: what changes when the accusative is absent? Write the sentence with the subject omitted (pro-drop) and explain what must be recovered from context.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" pandas`. Download the language resources once with `turkicnlp.download(lang, processors=[...])`, as shown in the first code cell. Cells that fetch data from the web need internet access.

In [ ]:
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

turkicnlp.download("tur", processors=["morph"])          # Apertium Turkish transducers (once)
tur_dir = ModelRegistry.get_model_path("tur", "morph", "apertium", script="Latn")

def load(path):
    stream = hfst.HfstInputStream(str(path)); fst = stream.read(); stream.close()
    fst.convert(hfst.ImplementationType.HFST_OLW_TYPE)   # optimized lookup: no epsilon symbols in output
    return fst

analyser = load(tur_dir / "tur.automorf.hfst")
generator = load(tur_dir / "tur.autogen.hfst")
analyse = lambda word: sorted({a for a, _ in analyser.lookup(word)})
generate = lambda analysis: sorted({f for f, _ in generator.lookup(analysis)})

In [ ]:
# Readings of the object noun: case and possessive analyses
for word in ["kitabı", "kitap"]:
    print(f"{word}: {analyse(word)}")

In [ ]:
# How a trained parser analyses the three sentences (Stanza model for Turkish IMST)
turkicnlp.download("tur", processors=["tokenize", "pos", "lemma", "depparse"])
nlp = turkicnlp.Pipeline("tur", processors=["tokenize", "pos", "lemma", "depparse"])
for sentence in ["Kitabı Ayşe okudu.", "Ayşe kitabı okudu.", "Ayşe kitap okudu.", "Kitabı okudu."]:
    print(sentence)
    for w in nlp(sentence).words:
        case = next((f for f in (w.feats or "").split("|") if f.startswith("Case=")), "")
        print(f"   {w.text:<8} {w.upos:<6} {case:<10} head={w.head} {w.deprel}")

## Worked solution

**Translations.** Both *Kitabı Ayşe okudu* and *Ayşe kitabı okudu* mean 'Ayşe read the book'. The accusative *-ı* marks *kitabı* as the specific direct object in both orders, so the grammatical relations do not change; the information structure does.

**Questions.** *Ayşe kitabı okudu* (SOV, neutral) answers *Ayşe ne yaptı?* 'What did Ayşe do?'. *Kitabı Ayşe okudu* (OSV) answers *Kitabı kim okudu?* 'Who read the book?': the answer, *Ayşe*, stands in the focus position directly before the verb.

**Without the accusative.** *Ayşe kitap okudu* means 'Ayşe read a book / did some book-reading'. The bare object is non-specific and normally stays directly before the verb; *Kitap Ayşe okudu* would not express the same meaning.

**Pro-drop.** *Kitabı okudu* '(She/he) read the book'. The verb *okudu* has no overt person suffix in the third-person singular, so the subject (Ayşe or someone else) must be recovered from the preceding context; even the person is not overtly marked.

**What the code shows.** Apertium returns both readings for *kitabı*: accusative and third-person possessive ('his/her book'). The parser analyses *kitabı* as Case=Acc in *Ayşe kitabı okudu*, but sentence-initial *Kitabı* as nominative with a possessive feature, and still labels it the object — the ambiguity the chapter describes, here also affected by capitalisation. Check such analyses before relying on them.